# Pipeline Mesh — Shadow Art Revisited

Optimización de malla 3D mediante renderizado diferenciable.
Mirror mode desactivado para comparabilidad directa con el pipeline voxel.

In [ ]:
import os, sys, random
import numpy as np
import torch
import cv2
import imageio
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from tqdm import tqdm
from skimage import img_as_ubyte

from pytorch3d.utils import ico_sphere
from pytorch3d.io import load_obj, save_obj
from pytorch3d.structures import Meshes
from pytorch3d.transforms import Rotate, Translate
from pytorch3d.renderer import (
    FoVOrthographicCameras, FoVPerspectiveCameras,
    look_at_view_transform, look_at_rotation,
    RasterizationSettings, MeshRenderer, MeshRasterizer, BlendParams,
    SoftSilhouetteShader, HardPhongShader, PointLights, TexturesVertex,
)
from pytorch3d.loss import (
    mesh_edge_loss, mesh_laplacian_smoothing, mesh_normal_consistency,
)

# ── rutas de datos
CUSTOM_PATH  = "/mnt/c/Users/xenon/OneDrive/Escritorio/Proyecto-grafica/data/custom_silhouettes/processed"
DATASET_PATH = "/mnt/c/Users/xenon/OneDrive/Escritorio/Proyecto-grafica/data/viewsdataset"
ROOT_DIR     = "/mnt/c/Users/xenon/OneDrive/Escritorio/Proyecto-grafica/mesh_results/"

os.makedirs(ROOT_DIR, exist_ok=True)
print("Paths OK")
print("CUDA:", torch.cuda.is_available())


In [ ]:
# ── Funciones de utilería (de utills.py) ────────────────────────────────────

def get_silhouette_renderer(device, cameras, image_size):
    blend_params = BlendParams(sigma=1e-4, gamma=1e-4)
    raster_settings = RasterizationSettings(
        image_size=image_size,
        blur_radius=np.log(1. / 1e-4 - 1.) * blend_params.sigma,
        faces_per_pixel=100,
    )
    return MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=SoftSilhouetteShader(blend_params=blend_params)
    )

def get_phong_renderer(device, cameras, image_size):
    raster_settings = RasterizationSettings(image_size=image_size, blur_radius=0.0, faces_per_pixel=1)
    lights = PointLights(device=device, location=((2.0, 2.0, -2.0),))
    return MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=HardPhongShader(device=device, cameras=cameras, lights=lights)
    )

def get_props(img, img_size=(256, 256)):
    rows, cols = img_size
    h0, w0 = int(rows * 0.7), int(cols * 0.7)
    cx0, cy0 = img_size[0] // 2, img_size[1] // 2
    conts, _ = cv2.findContours(img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    conts = sorted(conts, key=lambda c: cv2.contourArea(c), reverse=True)
    cnt = conts[0]
    x, y, w, h = cv2.boundingRect(cnt)
    cx1, cy1 = x + w / 2, y + h / 2
    sx, sy = w0 / w, h0 / h
    tx = cx0 * (1 - sx) + (cx0 - cx1)
    ty = cy0 * (1 - sy) + (cy0 - cy1)
    M = np.float32([[sx, 0, tx], [0, sy, ty]])
    return cv2.warpAffine(img, M, img_size), len(conts) - 1

def create_cameras(num_views, device, zdist, camera_mode="ortho", mirror_mode=False):
    if mirror_mode:
        elev = torch.linspace(0, 0, num_views * 2)
        azim = torch.linspace(0, 360 - 360 // (num_views * 2), num_views * 2)
    else:
        elev = torch.linspace(0, 0, num_views)
        azim = torch.linspace(0, 180 - 180 // num_views, num_views)
    R, T = look_at_view_transform(dist=zdist, elev=elev, azim=azim)
    cameras = FoVOrthographicCameras(device=device, R=R, T=T)
    return cameras, R, T

def load_silhouettes(files, img_size, device, mirror_mode=False):
    """Carga y preprocesa siluetas. files: lista de rutas absolutas."""
    images, tensors = [], []
    all_files = files + (files if mirror_mode else [])
    for i, f in enumerate(all_files):
        img_raw = cv2.imread(f, 0)
        if img_raw is None:
            raise FileNotFoundError(f"No se encontró: {f}")
        _, img = cv2.threshold(img_raw, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        img = img.astype(np.uint8)
        img = cv2.resize(img, img_size)
        if i >= len(files):  # imagen espejada
            img = cv2.flip(img, 1)
        img, _ = get_props(img, img_size)
        images.append(img)
        tensors.append(torch.from_numpy(img / 255.0).to(device))
    return images, tensors

print("Utilidades cargadas.")


In [ ]:
# ── Funciones de pérdida (de losses.py) ─────────────────────────────────────
import torch.nn.functional as F
from torch.autograd import Variable
from math import exp

def gaussian(window_size, sigma):
    g = torch.Tensor([exp(-(x - window_size//2)**2 / float(2*sigma**2)) for x in range(window_size)])
    return g / g.sum()

def create_window(window_size, sigma, channel):
    _1D = gaussian(window_size, sigma).unsqueeze(1)
    _2D = _1D.mm(_1D.t()).float().unsqueeze(0).unsqueeze(0)
    return Variable(_2D.expand(channel, 1, window_size, window_size).contiguous())

class MS_SSIM(torch.nn.Module):
    def __init__(self, device):
        super().__init__()
        self.channel = 1
        self.device = device

    def _ssim(self, img1, img2):
        _, c, w, h = img1.size()
        ws = min(w, h, 11)
        sigma = 1.5 * ws / 11
        window = create_window(ws, sigma, self.channel).to(self.device)
        mu1 = F.conv2d(img1, window, padding=ws//2, groups=self.channel)
        mu2 = F.conv2d(img2, window, padding=ws//2, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1**2, mu2**2, mu1*mu2
        s1 = F.conv2d(img1*img1, window, padding=ws//2, groups=self.channel) - mu1_sq
        s2 = F.conv2d(img2*img2, window, padding=ws//2, groups=self.channel) - mu2_sq
        s12 = F.conv2d(img1*img2, window, padding=ws//2, groups=self.channel) - mu1_mu2
        C1, C2 = (0.01*255)**2, (0.03*255)**2
        V1, V2 = 2*s12 + C2, s1 + s2 + C2
        return ((2*mu1_mu2+C1)*V1/((mu1_sq+mu2_sq+C1)*V2)).mean(), (V1/V2).mean()

    def forward(self, img1, img2, levels=5):
        weight = torch.Tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333]).to(self.device)
        msssim = torch.zeros(levels).to(self.device)
        mcs    = torch.zeros(levels).to(self.device)
        for i in range(levels):
            msssim[i], mcs[i] = self._ssim(img1, img2)
            img1 = F.avg_pool2d(img1, 2, 2)
            img2 = F.avg_pool2d(img2, 2, 2)
        return torch.clamp((torch.prod(mcs[:levels-1]**weight[:levels-1]) * msssim[levels-1]**weight[levels-1]) / 2, 0, 1)

def update_mesh_shape_prior_losses(mesh, loss):
    loss["edge"]      = mesh_edge_loss(mesh)
    loss["normal"]    = mesh_normal_consistency(mesh)
    loss["laplacian"] = mesh_laplacian_smoothing(mesh, method="uniform")

def silh_loss(pred, gt): return ((pred - gt) ** 2).mean()
def l1_loss(pred, gt):   return torch.abs(pred - gt).mean()
def iou_np(pred, gt):
    n = np.sum(pred * gt);  d = np.sum(pred + gt - pred * gt)
    return n / d if d > 0 else 1.0
def dice_np(pred, gt):
    n = np.sum(pred * gt);  d = np.sum(pred + gt)
    return 2*n / d if d > 0 else 1.0

print("Pérdidas cargadas.")


In [ ]:
# ── Hiperparámetros globales ─────────────────────────────────────────────────
device       = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MIRROR_MODE  = False   # igual que en el pipeline voxel
CAMERA_MODE  = "ortho"
IMG_SIZE     = 512
ZDIST        = 2.5
NITER        = 2000
LR           = 0.15
SILH_WT      = 1.6
L1_WT        = 1.6
MS_SSIM_WT   = 0.0
EDGE_WT      = 1.6
NORM_WT      = 0.6
LAPL_WT      = 1.2

ms_ssim_loss = MS_SSIM(device)
random.seed(43)
torch.cuda.set_device(device)
print("Device:", device)


In [ ]:
# ── Función de entrenamiento ─────────────────────────────────────────────────
def train(sub_exp_id, shadow_files, niter=NITER, lr=LR):
    global device

    num_views = len(shadow_files)
    cameras, Rs, Ts = create_cameras(num_views, device, ZDIST, CAMERA_MODE, MIRROR_MODE)

    silhouette_renderer = get_silhouette_renderer(device, FoVOrthographicCameras(device=device), IMG_SIZE)
    phong_renderer      = get_phong_renderer(device, FoVOrthographicCameras(device=device), IMG_SIZE)

    out_dir = os.path.join(ROOT_DIR, sub_exp_id)
    os.makedirs(out_dir, exist_ok=True)

    gif_path = os.path.join(out_dir, "sample_0_vid.gif")
    writer   = imageio.get_writer(gif_path, mode='I', duration=0.1)

    silhs, silhs_tensors = load_silhouettes(shadow_files, (IMG_SIZE, IMG_SIZE), device, MIRROR_MODE)

    losses_cfg = {
        "silhouette": {"weight": SILH_WT,   "values": []},
        "l1":         {"weight": L1_WT,     "values": []},
        "ms_ssim":    {"weight": MS_SSIM_WT,"values": []},
        "edge":       {"weight": EDGE_WT,   "values": []},
        "normal":     {"weight": NORM_WT,   "values": []},
        "laplacian":  {"weight": LAPL_WT,   "values": []},
    }

    src_mesh = ico_sphere(4, device)
    verts    = src_mesh.verts_packed().type(torch.float)
    faces    = src_mesh.faces_packed()
    verts_shape   = verts.shape
    deform_verts  = torch.full(verts_shape, 0.0, device=device, requires_grad=True)
    optimizer     = torch.optim.SGD([deform_verts], lr=lr, momentum=0.9)

    ms_ssim_list = []
    loop = tqdm(range(niter), desc=sub_exp_id)

    for i in loop:
        optimizer.zero_grad()
        loss = {k: torch.tensor(0.0, device=device) for k in losses_cfg}

        n_silh = torch.tensor(0.0, device=device)
        n_l1   = torch.tensor(0.0, device=device)
        n_ms   = torch.tensor(0.0, device=device)

        total_views = num_views * (1 + int(MIRROR_MODE))

        for n, silh_gt in enumerate(silhs_tensors):
            new_mesh = src_mesh.offset_verts(deform_verts)
            update_mesh_shape_prior_losses(new_mesh, loss)
            R = Rs[n].unsqueeze(0).to(device)
            T = Ts[n].unsqueeze(0).to(device)
            view_pred = silhouette_renderer(new_mesh, R=R, T=T)
            silh_pred = view_pred[..., 3][0]
            n_silh += silh_loss(silh_pred, silh_gt)
            n_l1   += l1_loss(silh_pred, silh_gt)
            n_ms   += ms_ssim_loss(
                silh_pred.view(1,1,IMG_SIZE,IMG_SIZE).float(),
                silh_gt  .view(1,1,IMG_SIZE,IMG_SIZE).float()
            )
            # guardar predicción
            pred_img = (silh_pred.detach().cpu().numpy() * 255).astype(np.uint8)
            _, pred_img = cv2.threshold(pred_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            cv2.imwrite(os.path.join(out_dir, f"sample_0_pred_view_{n}.png"), pred_img)

        loss["silhouette"] = n_silh / total_views
        loss["l1"]         = n_l1   / total_views
        loss["ms_ssim"]    = n_ms   / total_views

        sum_loss = sum(loss[k] * losses_cfg[k]["weight"] for k in losses_cfg)
        for k in losses_cfg:
            losses_cfg[k]["values"].append(loss[k].item())

        sum_loss.backward()
        optimizer.step()
        ms_ssim_list.append((n_ms / total_views).item())
        loop.set_postfix(loss=f"{sum_loss.item():.4f}")

        if sum_loss.item() < 1e-5:
            break

        if i % 10 == 0:
            new_mesh_vis = src_mesh.offset_verts(deform_verts)
            nv = new_mesh_vis.verts_packed()
            nf = new_mesh_vis.faces_packed()
            rgb = torch.ones_like(nv)[None]
            rgb[0,:,2] *= 0.4
            new_mesh_vis = Meshes(verts=[nv], faces=[nf],
                                  textures=TexturesVertex(verts_features=rgb.to(device)))
            R_gif, T_gif = look_at_view_transform(ZDIST, 0, i, device=device)
            img_gif = phong_renderer(new_mesh_vis, R=R_gif, T=T_gif)
            img_gif = img_as_ubyte(img_gif[0, ..., :3].detach().cpu().numpy())
            writer.append_data(img_gif)

    writer.close()

    final_mesh  = src_mesh.offset_verts(deform_verts)
    final_verts, final_faces = final_mesh.get_mesh_verts_faces(0)
    save_obj(os.path.join(out_dir, "sample_0_output.obj"), final_verts, final_faces)

    for idx, img in enumerate(silhs):
        cv2.imwrite(os.path.join(out_dir, f"sample_0view_{idx}.png"), img)

    ms_metric = 1 - np.mean(ms_ssim_list)
    mean_iou, mean_dice = 0.0, 0.0
    for idx in range(total_views):
        gt   = cv2.imread(os.path.join(out_dir, f"sample_0view_{idx}.png"), 0)
        pred = cv2.imread(os.path.join(out_dir, f"sample_0_pred_view_{idx}.png"), 0)
        if gt is None or pred is None: continue
        mean_iou  += iou_np(gt/255.0, pred/255.0)
        mean_dice += dice_np(gt/255.0, pred/255.0)
    mean_iou  /= total_views
    mean_dice /= total_views

    log = (
        f"Edge loss     : {mesh_edge_loss(final_mesh).item():.6f}\n"
        f"Laplacian loss: {mesh_laplacian_smoothing(final_mesh).item():.6f}\n"
        f"Normal loss   : {mesh_normal_consistency(final_mesh).item():.6f}\n"
        f"IOU           : {mean_iou:.6f}\n"
        f"Dice          : {mean_dice:.6f}\n"
        f"MISSIM        : {ms_metric:.6f}\n"
    )
    with open(os.path.join(out_dir, "log.txt"), "w") as f:
        f.write(log)
    print(log)
    return mean_iou, mean_dice

print("Función train() lista.")


In [ ]:
# ── Lista de experimentos (idéntica al pipeline voxel) ───────────────────────
experimentos = [
    # Siluetas propias — por complejidad
    {"output": "2-simple-simple",       "images": ["s1_circulo.png",     "s3_cruz.png"]},
    {"output": "2-simple-alta",         "images": ["s2_rectangulo.png",  "h2_running_person.png"]},
    {"output": "2-alta-alta",           "images": ["h3_snowflake.png",   "h5_bike.png"]},
    {"output": "5-simple",              "images": ["s1_circulo.png", "s2_rectangulo.png", "s3_cruz.png", "s4_flecha.png", "s5_house.png"]},
    {"output": "5-media",               "images": ["m1_bird.png", "m2_tree.png", "m3_fish.png", "m4_star.png", "m5_guitar.png"]},
    {"output": "5-alta",                "images": ["h1_sitting_cat.png", "h2_running_person.png", "h3_snowflake.png", "h4_butterfly.png", "h5_bike.png"]},
    {"output": "5-2simple-2media-1alta","images": ["s1_circulo.png", "s5_house.png", "m1_bird.png", "m5_guitar.png", "h3_snowflake.png"]},
    {"output": "10-5simple-5media",     "images": ["s1_circulo.png","s2_rectangulo.png","s3_cruz.png","s4_flecha.png","s5_house.png",
                                                    "m1_bird.png","m2_tree.png","m3_fish.png","m4_star.png","m5_guitar.png"]},
    {"output": "10-5media-5alta",       "images": ["m1_bird.png","m2_tree.png","m3_fish.png","m4_star.png","m5_guitar.png",
                                                    "h1_sitting_cat.png","h2_running_person.png","h3_snowflake.png","h4_butterfly.png","h5_bike.png"]},
    # Baseline — dataset original del paper
    {"output": "2-mikey-puma",          "images": ["mikey.png",    "puma.png"],                              "dataset": "original"},
    {"output": "3-heroes",              "images": ["Spider-Man.png","superman.png","batman.png"],             "dataset": "original"},
    {"output": "3-bunny-teddy-duck",    "images": ["bunny_0.png",  "teddy2.png","duck.png"],                 "dataset": "original"},
    {"output": "3-mikey-puma-heart",    "images": ["mikey.png",    "puma.png","heart.png"],                  "dataset": "original"},
]

def resolve_paths(images, dataset="custom"):
    base = DATASET_PATH if dataset == "original" else CUSTOM_PATH
    return [os.path.join(base, img) for img in images]

print(f"Total experimentos: {len(experimentos)}")


In [ ]:
# ── Ejecutar todos los experimentos ─────────────────────────────────────────
# Para correr solo algunos, comenta los que no quieras o ajusta el slice:
# experimentos_a_correr = experimentos[:3]

for experimento in experimentos:
    print("\n" + "="*60)
    print(f" EXPERIMENTO: {experimento['output']}")
    print(f" SILUETAS:    {experimento['images']}")
    print("="*60)

    paths = resolve_paths(experimento["images"], experimento.get("dataset", "custom"))
    train(experimento["output"], paths)

print("\n✅ Todos los experimentos finalizados.")


In [ ]:
# ── Cálculo de métricas consolidadas ────────────────────────────────────────
import pandas as pd

def _roi_pixel_accuracy(gt_bool, pred_bool, margin=5):
    def bbox(m):
        ys, xs = np.where(m)
        return (xs.min(), xs.max(), ys.min(), ys.max()) if len(xs) > 0 else None
    b_gt, b_pred = bbox(gt_bool), bbox(pred_bool)
    if b_gt is None and b_pred is None: return 1.0
    b_gt = b_gt or b_pred;  b_pred = b_pred or b_gt
    xmin = max(0, min(b_gt[0], b_pred[0]) - margin)
    xmax = min(gt_bool.shape[1]-1, max(b_gt[1], b_pred[1]) + margin)
    ymin = max(0, min(b_gt[2], b_pred[2]) - margin)
    ymax = min(gt_bool.shape[0]-1, max(b_gt[3], b_pred[3]) + margin)
    return float(np.mean(gt_bool[ymin:ymax+1, xmin:xmax+1] == pred_bool[ymin:ymax+1, xmin:xmax+1]))

def compute_all_metrics_mesh(exp_id, num_views):
    """Calcula métricas por vista para un experimento mesh.
    Usa sample_0_pred_view_N.png como predicción (formato mesh).
    """
    folder  = os.path.join(ROOT_DIR, exp_id)
    results = []
    total   = num_views * (1 + int(MIRROR_MODE))

    for idx in range(total):
        gt_path   = os.path.join(folder, f"sample_0view_{idx}.png")
        pred_path = os.path.join(folder, f"sample_0_pred_view_{idx}.png")

        gt   = cv2.imread(gt_path,   0)
        pred = cv2.imread(pred_path, 0)
        if gt is None or pred is None:
            print(f"  Falta archivo para vista {idx}, saltando.")
            continue

        _, gt_bin   = cv2.threshold(gt,   127, 255, cv2.THRESH_BINARY)
        _, pred_bin = cv2.threshold(pred, 127, 255, cv2.THRESH_BINARY)
        gt_n, pred_n = gt_bin / 255.0, pred_bin / 255.0

        if np.sum(gt_n) == 0 and np.sum(pred_n) == 0:
            iou_val, dice_val = 1.0, 1.0
        else:
            iou_val  = float(iou_np(pred_n, gt_n))
            dice_val = float(dice_np(pred_n, gt_n))

        gt_b, pred_b = gt_bin > 0, pred_bin > 0
        tp = np.logical_and( gt_b,  pred_b).sum()
        fp = np.logical_and(~gt_b,  pred_b).sum()
        fn = np.logical_and( gt_b, ~pred_b).sum()

        precision = float(tp / (tp + fp)) if (tp + fp) > 0 else 1.0
        recall    = float(tp / (tp + fn)) if (tp + fn) > 0 else 1.0
        roi_acc   = _roi_pixel_accuracy(gt_b, pred_b)

        results.append({"view": idx, "IoU": iou_val, "Dice": dice_val,
                         "Precision": precision, "Recall": recall, "ROI_PA": roi_acc})
    return results

# ── Evaluar todos los experimentos
experimentos_eval = [
    {"output": "2-simple-simple",        "num_views": 2},
    {"output": "2-simple-alta",          "num_views": 2},
    {"output": "2-alta-alta",            "num_views": 2},
    {"output": "5-simple",               "num_views": 5},
    {"output": "5-media",                "num_views": 5},
    {"output": "5-alta",                 "num_views": 5},
    {"output": "5-2simple-2media-1alta", "num_views": 5},
    {"output": "10-5simple-5media",      "num_views": 10},
    {"output": "10-5media-5alta",        "num_views": 10},
    {"output": "2-mikey-puma",           "num_views": 2},
    {"output": "3-heroes",               "num_views": 3},
    {"output": "3-bunny-teddy-duck",     "num_views": 3},
    {"output": "3-mikey-puma-heart",     "num_views": 3},
]

todos = []
for experimento in experimentos_eval:
    folder = os.path.join(ROOT_DIR, experimento["output"])
    if not os.path.exists(folder):
        print(f"⚠️  {experimento['output']} — carpeta no encontrada, saltando.")
        continue
    print(f"\n{'='*60}\n {experimento['output'].upper()}\n{'='*60}")
    results = compute_all_metrics_mesh(experimento["output"], experimento["num_views"])
    for r in results:
        print(f"  Vista {r['view']}: IoU={r['IoU']:.4f}  Dice={r['Dice']:.4f}  "
              f"Prec={r['Precision']:.4f}  Rec={r['Recall']:.4f}  ROI={r['ROI_PA']:.4f}")
        todos.append({"experimento": experimento["output"], **r})

if todos:
    df = pd.DataFrame(todos)[["experimento","view","IoU","Dice","Precision","Recall","ROI_PA"]]
    csv_path = os.path.join(ROOT_DIR, "metricas_consolidadas.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n✅ Métricas exportadas a: {csv_path}")
    display(df.groupby("experimento")[["IoU","Dice","Precision","Recall","ROI_PA"]].mean().round(4))


In [ ]:
# ── Visualización 3D del mesh optimizado ────────────────────────────────────
import trimesh

def visualize_mesh(exp_id):
    obj_path = os.path.join(ROOT_DIR, exp_id, "sample_0_output.obj")
    if not os.path.exists(obj_path):
        print(f"No encontrado: {obj_path}")
        return
    mesh = trimesh.load(obj_path, force='mesh')
    v, f = np.array(mesh.vertices), np.array(mesh.faces)

    fig = plt.figure(figsize=(16, 4))
    fig.patch.set_facecolor('#0e1117')
    for i, (title, elev, azim) in enumerate([
        ('Frontal', 0, 0), ('Lateral', 0, 90), ('Superior', 90, 0), ('3/4', 25, 45)
    ]):
        ax = fig.add_subplot(1, 4, i+1, projection='3d')
        ax.set_facecolor('#0e1117')
        poly = Poly3DCollection(v[f], alpha=0.6, facecolor='steelblue', edgecolor='none')
        ax.add_collection3d(poly)
        ax.auto_scale_xyz(v[:,0], v[:,1], v[:,2])
        ax.view_init(elev=elev, azim=azim)
        ax.axis('off')
        ax.set_title(title, color='white', fontsize=9)
    fig.suptitle(f'Mesh — {exp_id}', color='white', fontsize=12)
    plt.tight_layout()
    plt.show()

# Ejemplo: visualizar el primer experimento que haya corrido
visualize_mesh("2-mikey-puma")
